In [ ]:
from pathlib import Path

import pandas as pd
import scipy.sparse as sp
import networkx as nx


# ============================================================
# PATHS
# ============================================================

ATHENA_DIR = Path(
    "../../../data/athena"
)
OUTPUT_DIR = Path("../../output")


# ============================================================
# LOAD CELL DATA
# ============================================================

cell_df = pd.read_csv(
    ATHENA_DIR / "cell_df" / "TENX68_cells.csv"
)


# ============================================================
# LOAD WHOLE-SLIDE kNN GRAPH
# ============================================================

graph_matrix = sp.load_npz(
    ATHENA_DIR / "cell_graphs" / "TENX68_knn_k5.npz"
)

cell_graph = nx.from_scipy_sparse_array(graph_matrix)


# ============================================================
# VALIDATION
# ============================================================

print("cell_df shape:", cell_df.shape)
print("Graph nodes:", cell_graph.number_of_nodes())
print("Graph edges:", cell_graph.number_of_edges())
print("Graph shape:", graph_matrix.shape)

print("node_index minimum:", cell_df["node_index"].min())
print("node_index maximum:", cell_df["node_index"].max())
print("Unique node indices:", cell_df["node_index"].nunique())

In [ ]:
import networkx as nx


# ============================================================
# SETTINGS
# ============================================================

WINDOW_ID = "TENX68_r2_c8"
PHENOTYPE_COL = "cell_type"
NODE_COL = "node_index"


# ============================================================
# 1. EXTRACT CELLS FROM THE SELECTED WINDOW
# ============================================================

window_cells = (
    cell_df.loc[cell_df["window_id"] == WINDOW_ID]
    .copy()
    .reset_index(drop=True)
)

if window_cells.empty:
    raise ValueError(f"No cells found for window: {WINDOW_ID}")


# ============================================================
# 2. EXTRACT THE CORRESPONDING WINDOW SUBGRAPH
# ============================================================

window_node_ids = window_cells[NODE_COL].tolist()

window_subgraph = cell_graph.subgraph(window_node_ids).copy()


# ============================================================
# 3. ADD / VERIFY PHENOTYPE NODE ATTRIBUTES
# ============================================================

phenotype_map = dict(
    zip(
        window_cells[NODE_COL],
        window_cells[PHENOTYPE_COL],
    )
)

nx.set_node_attributes(
    window_subgraph,
    phenotype_map,
    PHENOTYPE_COL,
)


# ============================================================
# 4. BASIC VALIDATION
# ============================================================

missing_nodes = set(window_node_ids) - set(window_subgraph.nodes)

if missing_nodes:
    raise ValueError(
        f"{len(missing_nodes)} window cells are missing from the graph."
    )

nodes_without_phenotype = [
    node
    for node, data in window_subgraph.nodes(data=True)
    if PHENOTYPE_COL not in data
]

if nodes_without_phenotype:
    raise ValueError(
        f"{len(nodes_without_phenotype)} graph nodes have no phenotype."
    )


# ============================================================
# 5. GRAPH SUMMARY
# ============================================================

n_nodes = window_subgraph.number_of_nodes()
n_edges = window_subgraph.number_of_edges()
n_isolates = nx.number_of_isolates(window_subgraph)
n_components = nx.number_connected_components(window_subgraph)

phenotype_counts = (
    window_cells[PHENOTYPE_COL]
    .value_counts()
    .reindex(
        [
            "Connective",
            "Epithelial",
            "Inflammatory",
            "Neoplastic",
        ],
        fill_value=0,
    )
)


# ============================================================
# 6. SUMMARY
# ============================================================

print(f"Window: {WINDOW_ID}")
print(f"Nodes: {n_nodes}")
print(f"Edges: {n_edges}")
print(f"Connected components: {n_components}")
print(f"Isolated nodes: {n_isolates}")

print("\nPhenotype counts:")
print(phenotype_counts)

In [ ]:
import numpy as np
import pandas as pd
import networkx as nx


# ============================================================
# SETTINGS
# ============================================================

PHENOTYPES = [
    "Connective",
    "Epithelial",
    "Inflammatory",
    "Neoplastic",
]

PHENOTYPE_COL = "cell_type"
NODE_COL = "node_index"

N_PERMUTATIONS = 500
RANDOM_STATE = 42


# ============================================================
# 1. ALIGN GRAPH NODES WITH PHENOTYPE LABELS
# ============================================================

node_list = list(window_subgraph.nodes())

phenotype_map = dict(
    zip(
        window_cells[NODE_COL],
        window_cells[PHENOTYPE_COL],
    )
)

labels = np.array(
    [phenotype_map[node] for node in node_list]
)

if pd.isna(labels).any():
    raise ValueError("Some graph nodes do not have phenotype labels.")


# ============================================================
# 2. CREATE SPARSE ADJACENCY MATRIX
# ============================================================

adjacency = nx.to_scipy_sparse_array(
    window_subgraph,
    nodelist=node_list,
    format="csr",
    dtype=float,
)

degrees = np.asarray(adjacency.sum(axis=1)).ravel()

if np.any(degrees == 0):
    raise ValueError("The window graph contains isolated nodes.")


# ============================================================
# 3. PROPORTIONAL INTERACTION MATRIX
# ============================================================

def compute_proportional_matrix(
    adjacency,
    degrees,
    labels,
    phenotypes,
):
    """
    Compute directed proportional neighborhood interactions.

    Rows:
        Source phenotype.

    Columns:
        Neighbor phenotype.
    """

    n_cells = len(labels)
    n_phenotypes = len(phenotypes)

    phenotype_to_index = {
        phenotype: index
        for index, phenotype in enumerate(phenotypes)
    }

    # One-hot phenotype matrix:
    # rows = cells
    # columns = phenotypes
    one_hot = np.zeros(
        (n_cells, n_phenotypes),
        dtype=float,
    )

    for cell_index, phenotype in enumerate(labels):
        one_hot[
            cell_index,
            phenotype_to_index[phenotype],
        ] = 1.0

    # Number of neighbors from each phenotype
    neighbor_counts = adjacency @ one_hot

    # Convert counts into proportions for every source cell
    neighbor_proportions = (
        neighbor_counts
        / degrees[:, None]
    )

    interaction_matrix = np.full(
        (n_phenotypes, n_phenotypes),
        np.nan,
        dtype=float,
    )

    # Average neighbor proportions over cells
    # belonging to each source phenotype
    for source_index, source_phenotype in enumerate(phenotypes):

        source_mask = labels == source_phenotype

        if source_mask.sum() == 0:
            continue

        interaction_matrix[source_index, :] = (
            neighbor_proportions[source_mask]
            .mean(axis=0)
        )

    return interaction_matrix


# ============================================================
# 4. OBSERVED INTERACTION MATRIX
# ============================================================

observed_matrix = compute_proportional_matrix(
    adjacency=adjacency,
    degrees=degrees,
    labels=labels,
    phenotypes=PHENOTYPES,
)


# ============================================================
# 5. PERMUTATION-EXPECTED MATRIX
# ============================================================

rng = np.random.default_rng(RANDOM_STATE)

permuted_matrices = np.zeros(
    (
        N_PERMUTATIONS,
        len(PHENOTYPES),
        len(PHENOTYPES),
    ),
    dtype=float,
)

for permutation_index in range(N_PERMUTATIONS):

    permuted_labels = rng.permutation(labels)

    permuted_matrices[permutation_index] = (
        compute_proportional_matrix(
            adjacency=adjacency,
            degrees=degrees,
            labels=permuted_labels,
            phenotypes=PHENOTYPES,
        )
    )

expected_matrix = np.nanmean(
    permuted_matrices,
    axis=0,
)


# ============================================================
# 6. INTERACTION ENRICHMENT MATRIX
# ============================================================

enrichment_matrix = (
    observed_matrix
    - expected_matrix
)


# ============================================================
# 7. GLOBAL INFILTRATION SCORE
# ============================================================

def compute_infiltration_score(
    graph,
    phenotype_map,
    source_phenotype,
    target_phenotype,
):
    """
    ATHENA global infiltration score:

        source-target edges
        -------------------
        source-source edges

    Returns NaN when there are no source-source edges.
    """

    source_target_edges = 0
    source_source_edges = 0

    for node_u, node_v in graph.edges():

        phenotype_u = phenotype_map[node_u]
        phenotype_v = phenotype_map[node_v]

        pair = {phenotype_u, phenotype_v}

        if (
            source_phenotype != target_phenotype
            and pair == {source_phenotype, target_phenotype}
        ):
            source_target_edges += 1

        if (
            phenotype_u == source_phenotype
            and phenotype_v == source_phenotype
        ):
            source_source_edges += 1

    if source_source_edges == 0:
        infiltration_score = np.nan
    else:
        infiltration_score = (
            source_target_edges
            / source_source_edges
        )

    return {
        "score": infiltration_score,
        "source_target_edges": source_target_edges,
        "source_source_edges": source_source_edges,
    }


infiltration_result = compute_infiltration_score(
    graph=window_subgraph,
    phenotype_map=phenotype_map,
    source_phenotype="Inflammatory",
    target_phenotype="Neoplastic",
)


# ============================================================
# 8. CONVERT MATRICES TO DATAFRAMES
# ============================================================

observed_df = pd.DataFrame(
    observed_matrix,
    index=PHENOTYPES,
    columns=PHENOTYPES,
)

expected_df = pd.DataFrame(
    expected_matrix,
    index=PHENOTYPES,
    columns=PHENOTYPES,
)

enrichment_df = pd.DataFrame(
    enrichment_matrix,
    index=PHENOTYPES,
    columns=PHENOTYPES,
)

observed_df.index.name = "Source"
expected_df.index.name = "Source"
enrichment_df.index.name = "Source"


# ============================================================
# 9. SUMMARY
# ============================================================

print(f"Window: {WINDOW_ID}")
print(f"Permutations: {N_PERMUTATIONS}")

print("\nObserved proportional interactions:")
print(observed_df.round(4))

print("\nExpected random interactions:")
print(expected_df.round(4))

print("\nInteraction enrichment:")
print(enrichment_df.round(4))

print("\nInflammatory → Neoplastic infiltration:")
print(
    "Inflammatory-Neoplastic edges:",
    infiltration_result["source_target_edges"],
)
print(
    "Inflammatory-Inflammatory edges:",
    infiltration_result["source_source_edges"],
)
print(
    "Infiltration score:",
    infiltration_result["score"],
)

In [ ]:
print("Observed row sums:")
print(observed_df.sum(axis=1).round(6))

print("\nEnrichment row sums:")
print(enrichment_df.sum(axis=1).round(6))

In [ ]:
import numpy as np
import pandas as pd


# ============================================================
# 1. EXPECTED MEAN
# ============================================================

expected_matrix = np.nanmean(
    permuted_matrices,
    axis=0,
)

# ============================================================
# 2. STANDARD DEVIATION
# ============================================================

std_matrix = np.nanstd(
    permuted_matrices,
    axis=0,
)

# ============================================================
# 3. Z-SCORE
# ============================================================

z_matrix = (
    observed_matrix - expected_matrix
) / (std_matrix + 1e-12)


# ============================================================
# 4. EMPIRICAL P-VALUE
# ============================================================

p_matrix = np.ones_like(observed_matrix)

for i in range(len(PHENOTYPES)):
    for j in range(len(PHENOTYPES)):

        null = permuted_matrices[:, i, j]

        observed = observed_matrix[i, j]

        p = (
            np.sum(
                np.abs(null - null.mean())
                >= abs(observed - null.mean())
            )
            + 1
        ) / (N_PERMUTATIONS + 1)

        p_matrix[i, j] = p


# ============================================================
# 5. DATAFRAMES
# ============================================================

z_df = pd.DataFrame(
    z_matrix,
    index=PHENOTYPES,
    columns=PHENOTYPES,
)

p_df = pd.DataFrame(
    p_matrix,
    index=PHENOTYPES,
    columns=PHENOTYPES,
)


# ============================================================
# 6. PRINT RESULTS
# ============================================================

print("\nZ-score matrix")
print(z_df.round(2))

print("\nEmpirical p-value matrix")
print(p_df.round(4))

In [ ]:
import numpy as np
import pandas as pd
import networkx as nx


PHENOTYPES = [
    "Connective",
    "Epithelial",
    "Inflammatory",
    "Neoplastic",
]


# ============================================================
# PROPORTIONAL MATRIX FOR ONE LABEL ASSIGNMENT
# ============================================================

def _compute_proportional_matrix(
    adjacency,
    degrees,
    labels,
    phenotypes,
):
    """
    Compute directed proportional neighborhood interactions.

    Rows:
        Source phenotype.

    Columns:
        Neighbor phenotype.
    """

    n_cells = len(labels)
    n_phenotypes = len(phenotypes)

    phenotype_to_index = {
        phenotype: index
        for index, phenotype in enumerate(phenotypes)
    }

    one_hot = np.zeros(
        (n_cells, n_phenotypes),
        dtype=float,
    )

    for cell_index, phenotype in enumerate(labels):
        one_hot[
            cell_index,
            phenotype_to_index[phenotype],
        ] = 1.0

    # Number of neighbors belonging to each phenotype
    neighbor_counts = adjacency @ one_hot

    # Fraction of each cell's neighbors belonging to each phenotype
    neighbor_proportions = (
        neighbor_counts / degrees[:, None]
    )

    matrix = np.full(
        (n_phenotypes, n_phenotypes),
        np.nan,
        dtype=float,
    )

    for source_index, source_phenotype in enumerate(phenotypes):

        source_mask = labels == source_phenotype

        if source_mask.sum() == 0:
            continue

        matrix[source_index] = (
            neighbor_proportions[source_mask]
            .mean(axis=0)
        )

    return matrix


# ============================================================
# COMPLETE PROPORTIONAL INTERACTION ANALYSIS
# ============================================================

def compute_proportional_interactions(
    window_cells,
    window_subgraph,
    phenotype_col="cell_type",
    node_col="node_index",
    phenotypes=PHENOTYPES,
    n_permutations=500,
    random_state=42,
    compute_p_values=True,
    debug=False,
):
    """
    Compute ATHENA proportional neighborhood interactions.

    Returns
    -------
    dict
        observed:
            Observed proportional interaction DataFrame.

        expected:
            Mean proportional interaction DataFrame from
            randomized phenotype labels.

        enrichment:
            Observed minus expected interaction DataFrame.

        p_values:
            Empirical two-sided permutation p-value DataFrame.
            None when compute_p_values=False.

        permuted_matrices:
            Raw permutation matrices, shape
            (n_permutations, n_phenotypes, n_phenotypes).
    """

    if n_permutations < 1:
        raise ValueError(
            "n_permutations must be at least 1 because "
            "enrichment requires a randomized expected matrix."
        )

    required_columns = {node_col, phenotype_col}

    missing_columns = required_columns - set(window_cells.columns)

    if missing_columns:
        raise KeyError(
            f"Missing columns in window_cells: {sorted(missing_columns)}"
        )

    unknown_phenotypes = (
        set(window_cells[phenotype_col].dropna().unique())
        - set(phenotypes)
    )

    if unknown_phenotypes:
        raise ValueError(
            f"Unexpected phenotypes: {sorted(unknown_phenotypes)}"
        )

    # --------------------------------------------------------
    # Align cells with graph nodes
    # --------------------------------------------------------

    phenotype_map = dict(
        zip(
            window_cells[node_col],
            window_cells[phenotype_col],
        )
    )

    node_list = list(window_subgraph.nodes())

    missing_nodes = [
        node
        for node in node_list
        if node not in phenotype_map
    ]

    if missing_nodes:
        raise ValueError(
            f"{len(missing_nodes)} graph nodes are missing "
            "from window_cells."
        )

    labels = np.asarray(
        [phenotype_map[node] for node in node_list],
        dtype=object,
    )

    # --------------------------------------------------------
    # Sparse adjacency matrix
    # --------------------------------------------------------

    adjacency = nx.to_scipy_sparse_array(
        window_subgraph,
        nodelist=node_list,
        format="csr",
        dtype=float,
    )

    degrees = np.asarray(
        adjacency.sum(axis=1)
    ).ravel()

    if np.any(degrees == 0):
        raise ValueError(
            "The window graph contains isolated nodes."
        )

    # --------------------------------------------------------
    # Observed interaction matrix
    # --------------------------------------------------------

    observed_matrix = _compute_proportional_matrix(
        adjacency=adjacency,
        degrees=degrees,
        labels=labels,
        phenotypes=phenotypes,
    )

    # --------------------------------------------------------
    # Random phenotype-label permutations
    # --------------------------------------------------------

    rng = np.random.default_rng(random_state)

    permuted_matrices = np.empty(
        (
            n_permutations,
            len(phenotypes),
            len(phenotypes),
        ),
        dtype=float,
    )

    for permutation_index in range(n_permutations):

        permuted_labels = rng.permutation(labels)

        permuted_matrices[permutation_index] = (
            _compute_proportional_matrix(
                adjacency=adjacency,
                degrees=degrees,
                labels=permuted_labels,
                phenotypes=phenotypes,
            )
        )

    expected_matrix = np.nanmean(
        permuted_matrices,
        axis=0,
    )

    enrichment_matrix = (
        observed_matrix - expected_matrix
    )

    # --------------------------------------------------------
    # Optional empirical p-values
    # --------------------------------------------------------

    p_value_matrix = None

    if compute_p_values:

        p_value_matrix = np.full_like(
            observed_matrix,
            np.nan,
            dtype=float,
        )

        for source_index in range(len(phenotypes)):
            for target_index in range(len(phenotypes)):

                null_values = permuted_matrices[
                    :,
                    source_index,
                    target_index,
                ]

                null_mean = np.nanmean(null_values)

                observed_difference = abs(
                    observed_matrix[
                        source_index,
                        target_index,
                    ]
                    - null_mean
                )

                null_differences = abs(
                    null_values - null_mean
                )

                p_value_matrix[
                    source_index,
                    target_index,
                ] = (
                    np.sum(
                        null_differences
                        >= observed_difference
                    )
                    + 1
                ) / (n_permutations + 1)

    # --------------------------------------------------------
    # Convert to DataFrames
    # --------------------------------------------------------

    observed_df = pd.DataFrame(
        observed_matrix,
        index=phenotypes,
        columns=phenotypes,
    )

    expected_df = pd.DataFrame(
        expected_matrix,
        index=phenotypes,
        columns=phenotypes,
    )

    enrichment_df = pd.DataFrame(
        enrichment_matrix,
        index=phenotypes,
        columns=phenotypes,
    )

    p_values_df = None

    if p_value_matrix is not None:
        p_values_df = pd.DataFrame(
            p_value_matrix,
            index=phenotypes,
            columns=phenotypes,
        )

    for matrix_df in [
        observed_df,
        expected_df,
        enrichment_df,
        p_values_df,
    ]:
        if matrix_df is not None:
            matrix_df.index.name = "Source"

    if debug:
        print(
            f"Nodes: {window_subgraph.number_of_nodes()}"
        )
        print(
            f"Edges: {window_subgraph.number_of_edges()}"
        )
        print(
            f"Permutations: {n_permutations}"
        )
        print("\nInteraction enrichment:")
        print(enrichment_df.round(4))

    return {
        "observed": observed_df,
        "expected": expected_df,
        "enrichment": enrichment_df,
        "p_values": p_values_df,
        "permuted_matrices": permuted_matrices,
    }

In [ ]:
interaction_results = compute_proportional_interactions(
    window_cells=window_cells,
    window_subgraph=window_subgraph,
    n_permutations=500,
    random_state=42,
    compute_p_values=True,
    debug=True,
)

In [ ]:
observed_df = interaction_results["observed"]
expected_df = interaction_results["expected"]
enrichment_df = interaction_results["enrichment"]
p_values_df = interaction_results["p_values"]

In [ ]:
print("\nObserved:")
print(observed_df.round(4))

print("\nExpected:")
print(expected_df.round(4))

print("\nEnrichment:")
print(enrichment_df.round(4))

print("\nEmpirical p-values:")
print(p_values_df.round(4))

In [ ]:
interaction_results = compute_proportional_interactions(
    window_cells=window_cells,
    window_subgraph=window_subgraph,
    n_permutations=100,
    random_state=42,
    compute_p_values=False,
    debug=False,
)

enrichment_df = interaction_results["enrichment"]

In [ ]:
enrichment_df

In [ ]:
import matplotlib.pyplot as plt
import networkx as nx
from pathlib import Path


# ============================================================
# OUTPUT DIRECTORY
# ============================================================

output_dir = (
    OUTPUT_DIR
    / "figures"
    / "interaction"
)

output_dir.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# PHENOTYPE COLORS
# ============================================================

phenotype_colors = {
    "Connective": "#66c2a5",
    "Epithelial": "#fc8d62",
    "Inflammatory": "#8da0cb",
    "Neoplastic": "#e41a1c",
}


# ============================================================
# NODE POSITIONS
# ============================================================

position = dict(
    zip(
        window_cells["node_index"],
        zip(
            window_cells["x"],
            -window_cells["y"],      # invert y to match WSI orientation
        ),
    )
)


# ============================================================
# NODE COLORS
# ============================================================

node_colors = [
    phenotype_colors[
        window_subgraph.nodes[node]["cell_type"]
    ]
    for node in window_subgraph.nodes()
]


# ============================================================
# DRAW GRAPH
# ============================================================

plt.figure(figsize=(8,8))

nx.draw_networkx_edges(
    window_subgraph,
    pos=position,
    edge_color="lightgray",
    width=0.3,
    alpha=0.4,
)

nx.draw_networkx_nodes(
    window_subgraph,
    pos=position,
    node_color=node_colors,
    node_size=8,
)

plt.title(WINDOW_ID)

plt.axis("off")

plt.tight_layout()

plt.savefig(
    output_dir / f"{WINDOW_ID}_graph.png",
    dpi=300,
)

plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns


plt.figure(figsize=(6,5))

sns.heatmap(
    enrichment_df,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    square=True,
    linewidths=0.5,
    cbar_kws={
        "label": "Interaction enrichment"
    },
)

plt.title(
    f"{WINDOW_ID}\nInteraction enrichment"
)

plt.tight_layout()

plt.savefig(
    output_dir / f"{WINDOW_ID}_interaction_heatmap.png",
    dpi=300,
)

plt.show()

In [ ]:
# ============================================================
# Window-level interaction features
# ============================================================

import numpy as np
import pandas as pd
import networkx as nx


PHENOTYPES = [
    "Connective",
    "Epithelial",
    "Inflammatory",
    "Neoplastic",
]


# ============================================================
# PROPORTIONAL NEIGHBORHOOD INTERACTIONS
# ============================================================

def _proportional_matrix(
    adjacency,
    degrees,
    labels,
    phenotypes,
):
    phenotype_to_index = {
        phenotype: i
        for i, phenotype in enumerate(phenotypes)
    }

    one_hot = np.zeros(
        (len(labels), len(phenotypes)),
        dtype=float,
    )

    for cell_index, phenotype in enumerate(labels):
        one_hot[
            cell_index,
            phenotype_to_index[phenotype],
        ] = 1.0

    neighbor_counts = adjacency @ one_hot

    neighbor_proportions = (
        neighbor_counts / degrees[:, None]
    )

    matrix = np.full(
        (len(phenotypes), len(phenotypes)),
        np.nan,
        dtype=float,
    )

    for source_index, source_phenotype in enumerate(phenotypes):

        source_mask = labels == source_phenotype

        if source_mask.sum() == 0:
            continue

        matrix[source_index] = (
            neighbor_proportions[source_mask]
            .mean(axis=0)
        )

    return matrix


# ============================================================
# INFILTRATION MATRIX
# ============================================================

def _infiltration_matrix(
    graph,
    phenotype_map,
    phenotypes,
):
    """
    ATHENA infiltration:

        edges(source, target)
        ---------------------
        edges(source, source)

    Diagonal values are returned as NaN.
    """

    edge_counts = pd.DataFrame(
        0,
        index=phenotypes,
        columns=phenotypes,
        dtype=int,
    )

    for node_u, node_v in graph.edges():

        phenotype_u = phenotype_map[node_u]
        phenotype_v = phenotype_map[node_v]

        edge_counts.loc[
            phenotype_u,
            phenotype_v,
        ] += 1

        if phenotype_u != phenotype_v:
            edge_counts.loc[
                phenotype_v,
                phenotype_u,
            ] += 1

    infiltration = pd.DataFrame(
        np.nan,
        index=phenotypes,
        columns=phenotypes,
        dtype=float,
    )

    for source in phenotypes:

        denominator = edge_counts.loc[
            source,
            source,
        ]

        if denominator == 0:
            continue

        for target in phenotypes:

            if source == target:
                continue

            infiltration.loc[
                source,
                target,
            ] = (
                edge_counts.loc[source, target]
                / denominator
            )

    return infiltration, edge_counts


# ============================================================
# MAIN FEATURE EXTRACTION
# ============================================================

def extract_cell_interaction_features(
    slide_id,
    window_id,
    window_cells,
    window_subgraph,
    phenotype_col="cell_type",
    node_col="node_index",
    phenotypes=PHENOTYPES,
    n_permutations=100,
    random_state=42,
    debug=False,
):
    """
    Compute ATHENA cell-interaction features for one window.

    Returns
    -------
    dict
        slide_id
        window_id
        infiltration matrix
        observed neighborhood matrix
        expected neighborhood matrix
        enrichment matrix
        flattened window-level features
    """

    phenotype_map = dict(
        zip(
            window_cells[node_col],
            window_cells[phenotype_col],
        )
    )

    node_list = list(window_subgraph.nodes())

    labels = np.asarray(
        [phenotype_map[node] for node in node_list],
        dtype=object,
    )

    adjacency = nx.to_scipy_sparse_array(
        window_subgraph,
        nodelist=node_list,
        format="csr",
        dtype=float,
    )

    degrees = np.asarray(
        adjacency.sum(axis=1)
    ).ravel()

    if np.any(degrees == 0):
        raise ValueError(
            f"Window {window_id} contains isolated nodes."
        )

    # --------------------------------------------------------
    # Infiltration
    # --------------------------------------------------------

    infiltration_df, edge_counts_df = (
        _infiltration_matrix(
            graph=window_subgraph,
            phenotype_map=phenotype_map,
            phenotypes=phenotypes,
        )
    )

    # --------------------------------------------------------
    # Observed proportional interactions
    # --------------------------------------------------------

    observed_matrix = _proportional_matrix(
        adjacency=adjacency,
        degrees=degrees,
        labels=labels,
        phenotypes=phenotypes,
    )

    # --------------------------------------------------------
    # Random expectation
    # --------------------------------------------------------

    rng = np.random.default_rng(random_state)

    permuted_matrices = np.empty(
        (
            n_permutations,
            len(phenotypes),
            len(phenotypes),
        ),
        dtype=float,
    )

    for permutation_index in range(n_permutations):

        permuted_labels = rng.permutation(labels)

        permuted_matrices[permutation_index] = (
            _proportional_matrix(
                adjacency=adjacency,
                degrees=degrees,
                labels=permuted_labels,
                phenotypes=phenotypes,
            )
        )

    expected_matrix = np.nanmean(
        permuted_matrices,
        axis=0,
    )

    enrichment_matrix = (
        observed_matrix - expected_matrix
    )

    observed_df = pd.DataFrame(
        observed_matrix,
        index=phenotypes,
        columns=phenotypes,
    )

    expected_df = pd.DataFrame(
        expected_matrix,
        index=phenotypes,
        columns=phenotypes,
    )

    enrichment_df = pd.DataFrame(
        enrichment_matrix,
        index=phenotypes,
        columns=phenotypes,
    )

    # --------------------------------------------------------
    # Flatten features
    # --------------------------------------------------------

    features = {
        "slide_id": slide_id,
        "window_id": window_id,
    }

    for source in phenotypes:
        for target in phenotypes:

            source_name = source.lower()
            target_name = target.lower()

            features[
                f"interaction_{source_name}_to_{target_name}"
            ] = enrichment_df.loc[source, target]

            if source != target:
                features[
                    f"infiltration_{source_name}_to_{target_name}"
                ] = infiltration_df.loc[source, target]

        features[
            f"homotypic_edges_{source_name}"
        ] = edge_counts_df.loc[source, source]

    if debug:
        print(f"Window {window_id}")
        print(
            f"Nodes: {window_subgraph.number_of_nodes()}"
        )
        print(
            f"Edges: {window_subgraph.number_of_edges()}"
        )
        print("\nInfiltration matrix:")
        print(infiltration_df.round(4))
        print("\nInteraction enrichment:")
        print(enrichment_df.round(4))

    return {
        "slide_id": slide_id,
        "window_id": window_id,
        "features": features,
        "infiltration_matrix": infiltration_df,
        "edge_count_matrix": edge_counts_df,
        "observed_matrix": observed_df,
        "expected_matrix": expected_df,
        "enrichment_matrix": enrichment_df,
    }

In [ ]:
interaction_result = extract_cell_interaction_features(
    slide_id="TENX68",
    window_id=WINDOW_ID,
    window_cells=window_cells,
    window_subgraph=window_subgraph,
    n_permutations=500,
    random_state=42,
    debug=True,
)

In [ ]:
interaction_features = interaction_result["features"]

infiltration_df = interaction_result["infiltration_matrix"]

enrichment_df = interaction_result["enrichment_matrix"]

observed_df = interaction_result["observed_matrix"]

expected_df = interaction_result["expected_matrix"]
interaction_features

In [ ]:
interaction_features_df = pd.DataFrame(
    [interaction_features]
)

interaction_features_df.head()

In [ ]:
from pathlib import Path
import warnings

import pandas as pd
import scipy.sparse as sp
import networkx as nx


# ============================================================
# SETTINGS
# ============================================================

ATHENA_DIR = Path(
    "../../../data/athena"
)
OUTPUT_DIR = Path("../../output")

SLIDE_ID = "TENX68"


# ============================================================
# LOAD INPUTS
# ============================================================

cell_df = pd.read_csv(
    ATHENA_DIR / "cell_df" / f"{SLIDE_ID}_cells.csv"
)

window_df = pd.read_csv(
    ATHENA_DIR / "window_df" / f"{SLIDE_ID}_windows.csv"
)

graph_matrix = sp.load_npz(
    ATHENA_DIR / "cell_graphs" / f"{SLIDE_ID}_knn_k5.npz"
)

cell_graph = nx.from_scipy_sparse_array(
    graph_matrix
)


# ============================================================
# COMPUTE FEATURES FOR ALL WINDOWS
# ============================================================

feature_rows = []
skipped_windows = []

window_ids = (
    window_df["window_id"]
    .dropna()
    .unique()
)

for window_id in window_ids:

    window_cells = (
        cell_df.loc[
            cell_df["window_id"] == window_id
        ]
        .copy()
    )

    if window_cells.empty:
        skipped_windows.append({
            "window_id": window_id,
            "reason": "no cells",
        })
        continue

    node_ids = (
        window_cells["node_index"]
        .astype(int)
        .tolist()
    )

    window_subgraph = (
        cell_graph.subgraph(node_ids)
        .copy()
    )

    if window_subgraph.number_of_nodes() == 0:
        skipped_windows.append({
            "window_id": window_id,
            "reason": "no graph nodes",
        })
        continue

    # --------------------------------------------------------
    # Remove isolated nodes
    # --------------------------------------------------------

    isolated_nodes = list(
        nx.isolates(window_subgraph)
    )

    if isolated_nodes:
        window_subgraph.remove_nodes_from(
            isolated_nodes
        )

        window_cells = window_cells.loc[
            ~window_cells["node_index"].isin(
                isolated_nodes
            )
        ].copy()

    if window_subgraph.number_of_nodes() == 0:
        skipped_windows.append({
            "window_id": window_id,
            "reason": "only isolated nodes",
        })
        continue

    if window_subgraph.number_of_edges() == 0:
        skipped_windows.append({
            "window_id": window_id,
            "reason": "no graph edges",
        })
        continue

    try:

        # Missing phenotypes intentionally produce NaN rows.
        # Suppress only the corresponding NumPy warning.
        with warnings.catch_warnings():
            warnings.filterwarnings(
                "ignore",
                message="Mean of empty slice",
                category=RuntimeWarning,
            )

            result = extract_cell_interaction_features(
                slide_id=SLIDE_ID,
                window_id=window_id,
                window_cells=window_cells,
                window_subgraph=window_subgraph,
                n_permutations=100,
                random_state=42,
                debug=False,
            )

        feature_rows.append(
            result["features"]
        )

    except (ValueError, KeyError) as error:
        skipped_windows.append({
            "window_id": window_id,
            "reason": str(error),
        })


# ============================================================
# CREATE FEATURES_DF
# ============================================================

features_df = pd.DataFrame(
    feature_rows
)

if features_df.empty:
    raise ValueError(
        "No interaction features were successfully computed."
    )

features_df = (
    features_df
    .sort_values(["slide_id", "window_id"])
    .reset_index(drop=True)
)


# ============================================================
# SAVE FEATURES
# ============================================================

output_dir = (
    OUTPUT_DIR / "features_df"
)

output_dir.mkdir(
    parents=True,
    exist_ok=True,
)

output_path = (
    output_dir
    / f"{SLIDE_ID}_features.csv"
)

features_df.to_csv(
    output_path,
    index=False,
)


# ============================================================
# SAVE SKIPPED WINDOWS
# ============================================================

skipped_df = pd.DataFrame(
    skipped_windows
)

skipped_path = (
    output_dir
    / f"{SLIDE_ID}_interaction_skipped_windows.csv"
)

if not skipped_df.empty:
    skipped_df.to_csv(
        skipped_path,
        index=False,
    )


# ============================================================
# SUMMARY
# ============================================================

print("Windows requested:", len(window_ids))
print("Windows processed:", len(features_df))
print("Windows skipped:", len(skipped_df))
print("Feature table shape:", features_df.shape)
print("Saved:", output_path)

if not skipped_df.empty:
    print("\nSkipped-window reasons:")
    print(
        skipped_df["reason"]
        .value_counts()
    )

In [ ]:
print(
    features_df[
        [
            "slide_id",
            "window_id",
            "infiltration_inflammatory_to_neoplastic",
        ]
    ].head()
)

In [ ]:
# ============================================================
# Infiltration overlay on the WSI thumbnail
# ============================================================

from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from matplotlib.patches import Rectangle, Patch

import pyvips


def plot_slide_infiltration_overlay(
    window_df,
    features_df,
    slide_id,
    source_phenotype,
    target_phenotype,
    slide_image_path,
    output_dir,
    thumbnail_width=2000,
    alpha=0.60,
    clip_percentile=95,
    show_undefined=True,
    annotate=False,
):
    """
    Overlay window-level ATHENA infiltration scores on a WSI thumbnail.
    """

    source_name = source_phenotype.lower()
    target_name = target_phenotype.lower()

    feature_col = (
        f"infiltration_{source_name}_to_{target_name}"
    )

    required_cols = {
        "slide_id",
        "window_id",
        "xmin",
        "xmax",
        "ymin",
        "ymax",
    }

    missing_cols = required_cols - set(window_df.columns)

    if missing_cols:
        raise KeyError(
            f"Missing window_df columns: {sorted(missing_cols)}"
        )

    if feature_col not in features_df.columns:
        raise KeyError(
            f"Missing feature column: {feature_col}"
        )

    # ========================================================
    # MERGE WINDOW GEOMETRY AND FEATURES
    # ========================================================

    plot_df = (
        window_df.loc[
            window_df["slide_id"] == slide_id,
            [
                "slide_id",
                "window_id",
                "xmin",
                "xmax",
                "ymin",
                "ymax",
            ],
        ]
        .merge(
            features_df.loc[
                features_df["slide_id"] == slide_id,
                [
                    "slide_id",
                    "window_id",
                    feature_col,
                ],
            ],
            on=["slide_id", "window_id"],
            how="left",
            validate="one_to_one",
        )
        .copy()
    )

    if plot_df.empty:
        raise ValueError(
            f"No windows found for slide {slide_id}."
        )

    # ========================================================
    # LOAD WSI WITH PYVIPS
    # ========================================================

    slide_image_path = Path(slide_image_path)

    if not slide_image_path.exists():
        raise FileNotFoundError(slide_image_path)

    # Sequential mode reduces memory usage
    slide = pyvips.Image.new_from_file(
        str(slide_image_path),
        access="sequential",
    )

    full_width = slide.width
    full_height = slide.height

    scale = thumbnail_width / full_width

    thumbnail = slide.resize(scale)

    # Convert to 8-bit RGB
    if thumbnail.bands > 3:
        thumbnail = thumbnail[:3]

    if thumbnail.bands == 1:
        thumbnail = thumbnail.colourspace("srgb")

    if thumbnail.format != "uchar":
        thumbnail = thumbnail.cast("uchar")

    thumb_width = thumbnail.width
    thumb_height = thumbnail.height

    memory = thumbnail.write_to_memory()

    thumbnail_array = np.frombuffer(
        memory,
        dtype=np.uint8,
    ).reshape(
        thumb_height,
        thumb_width,
        thumbnail.bands,
    )

    scale_x = thumb_width / full_width
    scale_y = thumb_height / full_height

    # ========================================================
    # COLOR SCALE
    # ========================================================

    valid_values = (
        plot_df[feature_col]
        .replace([np.inf, -np.inf], np.nan)
        .dropna()
    )

    if valid_values.empty:
        raise ValueError(
            f"No defined values found for {feature_col}."
        )

    vmax = np.percentile(
        valid_values,
        clip_percentile,
    )

    if not np.isfinite(vmax) or vmax <= 0:
        vmax = valid_values.max()

    norm = Normalize(
        vmin=0,
        vmax=vmax,
        clip=True,
    )

    cmap = plt.get_cmap("viridis")

    # ========================================================
    # PLOT
    # ========================================================

    fig, ax = plt.subplots(
        figsize=(10, 12)
    )

    ax.imshow(
        thumbnail_array,
        origin="upper",
    )

    for row in plot_df.itertuples(index=False):

        x = float(row.xmin) * scale_x
        y = float(row.ymin) * scale_y

        width = (
            float(row.xmax) - float(row.xmin)
        ) * scale_x

        height = (
            float(row.ymax) - float(row.ymin)
        ) * scale_y

        value = getattr(row, feature_col)

        if np.isfinite(value):

            patch = Rectangle(
                (x, y),
                width,
                height,
                facecolor=cmap(norm(value)),
                edgecolor="white",
                linewidth=0.4,
                alpha=alpha,
            )

            ax.add_patch(patch)

            if annotate:
                ax.text(
                    x + width / 2,
                    y + height / 2,
                    f"{value:.2f}",
                    ha="center",
                    va="center",
                    fontsize=5,
                    color="black",
                )

        elif show_undefined:

            patch = Rectangle(
                (x, y),
                width,
                height,
                facecolor="lightgray",
                edgecolor="gray",
                linewidth=0.4,
                hatch="///",
                alpha=0.45,
            )

            ax.add_patch(patch)

    # ========================================================
    # COLORBAR AND LEGEND
    # ========================================================

    scalar_mappable = plt.cm.ScalarMappable(
        norm=norm,
        cmap=cmap,
    )

    scalar_mappable.set_array([])

    colorbar = fig.colorbar(
        scalar_mappable,
        ax=ax,
        fraction=0.035,
        pad=0.02,
    )

    colorbar.set_label(
        "ATHENA infiltration score"
    )

    if show_undefined:

        undefined_patch = Patch(
            facecolor="lightgray",
            edgecolor="gray",
            hatch="///",
            label="Undefined: no source-source edges",
        )

        ax.legend(
            handles=[undefined_patch],
            loc="lower right",
        )

    ax.set_xlim(0, thumb_width)
    ax.set_ylim(thumb_height, 0)
    ax.set_aspect("equal")

    ax.set_xlabel("Slide x-coordinate")
    ax.set_ylabel("Slide y-coordinate")

    ax.set_title(
        f"{slide_id}\n"
        f"{source_phenotype} → "
        f"{target_phenotype} infiltration"
    )

    # ========================================================
    # SAVE
    # ========================================================

    output_dir = Path(output_dir)
    output_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    output_path = (
        output_dir
        / (
            f"{slide_id}_"
            f"{source_name}_to_{target_name}_"
            "infiltration_overlay.png"
        )
    )

    fig.tight_layout()

    fig.savefig(
        output_path,
        dpi=300,
        bbox_inches="tight",
    )

    plt.show()
    plt.close(fig)

    print(
        "Full-resolution dimensions:",
        full_width,
        full_height,
    )

    print(
        "Thumbnail dimensions:",
        thumb_width,
        thumb_height,
    )

    print(
        "Defined windows:",
        valid_values.shape[0],
    )

    print(
        "Undefined windows:",
        plot_df[feature_col].isna().sum(),
    )

    print("Saved:", output_path)

    return output_path

In [ ]:
plot_slide_infiltration_overlay(
    window_df=window_df,
    features_df=features_df,
    slide_id="TENX68",
    source_phenotype="Inflammatory",
    target_phenotype="Neoplastic",
    slide_image_path=(
        "../../../data/breast/wsis/TENX68.tif"
    ),
    output_dir=(
        OUTPUT_DIR
        / "figures"
        / "infiltration"
    ),
    thumbnail_width=2000,
    alpha=0.60,
    clip_percentile=95,
    show_undefined=True,
    annotate=False,
)

In [ ]:
plot_slide_infiltration_overlay(
    window_df=window_df,
    features_df=features_df,
    slide_id="TENX68",
    source_phenotype="Neoplastic",
    target_phenotype="Inflammatory",
    slide_image_path=(
        "../../../data/breast/wsis/TENX68.tif"
    ),
    output_dir=(
        OUTPUT_DIR
        / "figures"
        / "infiltration"
    ),
    thumbnail_width=2000,
    alpha=0.60,
    clip_percentile=95,
    show_undefined=True,
    annotate=False,
)

In [ ]:
plot_slide_infiltration_overlay(
    window_df=window_df,
    features_df=features_df,
    slide_id="TENX68",
    source_phenotype="Neoplastic",
    target_phenotype="Connective",
    slide_image_path=(
        "../../../data/breast/wsis/TENX68.tif"
    ),
    output_dir=(
        OUTPUT_DIR
        / "figures"
        / "infiltration"
    ),
    thumbnail_width=2000,
    alpha=0.60,
    clip_percentile=95,
    show_undefined=True,
    annotate=False,
)

In [ ]:
plot_slide_infiltration_overlay(
    window_df=window_df,
    features_df=features_df,
    slide_id="TENX68",
    source_phenotype="Neoplastic",
    target_phenotype="Epithelial",
    slide_image_path=(
        "../../../data/breast/wsis/TENX68.tif"
    ),
    output_dir=(
        OUTPUT_DIR
        / "figures"
        / "infiltration"
    ),
    thumbnail_width=2000,
    alpha=0.60,
    clip_percentile=95,
    show_undefined=True,
    annotate=False,
)